In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
df = pd.read_csv("Dataset.csv")

In [10]:
print(df.head())
print(df.info())

   Restaurant ID         Restaurant Name  Country Code              City  \
0        6317637        Le Petit Souffle           162       Makati City   
1        6304287        Izakaya Kikufuji           162       Makati City   
2        6300002  Heat - Edsa Shangri-La           162  Mandaluyong City   
3        6318506                    Ooma           162  Mandaluyong City   
4        6314302             Sambo Kojin           162  Mandaluyong City   

                                             Address  \
0  Third Floor, Century City Mall, Kalayaan Avenu...   
1  Little Tokyo, 2277 Chino Roces Avenue, Legaspi...   
2  Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...   
3  Third Floor, Mega Fashion Hall, SM Megamall, O...   
4  Third Floor, Mega Atrium, SM Megamall, Ortigas...   

                                     Locality  \
0   Century City Mall, Poblacion, Makati City   
1  Little Tokyo, Legaspi Village, Makati City   
2  Edsa Shangri-La, Ortigas, Mandaluyong City   
3      SM 

In [12]:
df.fillna({
    'Cuisines': 'Unknown',
    'Average Cost for two': df['Average Cost for two'].median(),
    'Aggregate rating': df['Aggregate rating'].mean()
}, inplace=True)

In [14]:
df = df[['Restaurant Name', 'Cuisines', 'Average Cost for two', 'Aggregate rating']]

In [16]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Cuisines'])

In [18]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [20]:
def recommend_restaurants(preferred_cuisine, max_cost=1000, min_rating=3.5, top_n=5):
    
    filtered_df = df[
        (df['Average Cost for two'] <= max_cost) &
        (df['Aggregate rating'] >= min_rating)
    ]
    
    recommendations = filtered_df[
        filtered_df['Cuisines'].str.contains(preferred_cuisine, case=False, na=False)
    ]
    
    return recommendations[['Restaurant Name', 'Cuisines', 'Average Cost for two', 'Aggregate rating']].head(top_n)

In [22]:
result = recommend_restaurants(
    preferred_cuisine="North Indian",
    max_cost=800,
    min_rating=4.0,
    top_n=5
)

print(result)

     Restaurant Name                                Cuisines  \
572           Gazebo  Indian, North Indian, Mughlai, Biryani   
579        Via Delhi           Indian, North Indian, Chinese   
580     Punjab Grill                    Indian, North Indian   
587  Barbeque Nation                    Indian, North Indian   
596        SpiceKlub       Indian, North Indian, Street Food   

     Average Cost for two  Aggregate rating  
572                   120               4.0  
579                   100               4.0  
580                   330               4.9  
587                   150               4.5  
596                   150               4.4  


In [24]:
df['combined_features'] = df['Cuisines'] + " " + df['Average Cost for two'].astype(str)

# TF-IDF
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

# Similarity\\\
cosine_sim = cosine_similarity(tfidf_matrix)

In [26]:
def recommend_by_name(name, top_n=5):
    idx = df[df['Restaurant Name'] == name].index[0]
    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    restaurant_indices = [i[0] for i in scores[1:top_n+1]]
    
    return df.iloc[restaurant_indices][['Restaurant Name', 'Cuisines']]

In [28]:
print(result.describe())

       Average Cost for two  Aggregate rating
count              5.000000          5.000000
mean             170.000000          4.360000
std               91.923882          0.378153
min              100.000000          4.000000
25%              120.000000          4.000000
50%              150.000000          4.400000
75%              150.000000          4.500000
max              330.000000          4.900000
